In [ ]:

import pandas as pd
import numpy as np
import tqdm
import torch
from pykeen.triples import TriplesFactory
from pykeen.models import DistMult, ConvE, ComplEx, RotatE
from pykeen.training import SLCWATrainingLoop
from pykeen.losses import MarginRankingLoss
from pykeen.evaluation import RankBasedEvaluator
from pykeen.optimizers import Adagrad
from pykeen.stoppers import EarlyStopper
from sklearn.pipeline import make_pipeline 
from pykeen.trackers import JSONResultTracker
from torch.optim.lr_scheduler import ExponentialLR
from pykeen.hpo import hpo_pipeline
import json
import optuna

device = 'cuda' if torch.cuda.is_available() else 'cpu'


In [ ]:
main_data = pd.read_csv('data/clean_data/subgraphs/1.csv')
main_data = main_data.astype(str)
triples = main_data[['id_1', 'interaction', 'id_2']].values
triplet_data = TriplesFactory.from_labeled_triples(triples)
training_set, testing_set, validation_set = triplet_data.split([0.8, 0.1, 0.1], random_state=100)

In [11]:
def training_cycle(trial):
    
    embedding_dim = trial.suggest_int('embedding_dim', 128, 256, step = 32)
    lr = trial.suggest_float('lr', 1e-1, 5e-1)
    batch_size = trial.suggest_categorical('batch_size', [1024, 2048, 4096, 8192])
    num_epochs = 10
    margin = trial.suggest_float('margin', 1, 2)
    gamma = trial.suggest_float('gamma', 0.9, 1)

    loss_function = MarginRankingLoss(margin=margin)

    model = DistMult(
        triples_factory=training_set,
        embedding_dim = embedding_dim,
        random_seed = 100,
        loss = loss_function,
        )
    model = model.to(device)

    optimizer = Adagrad(params=model.get_grad_params(), 
                        lr=lr
                        )
    scheduler = ExponentialLR(optimizer, 
                              gamma=gamma
                              )
    #result_tracker = JSONResultTracker(path="pykeen/results.json")

    training_loop = SLCWATrainingLoop(
        model=model,
        triples_factory=training_set,
        optimizer=optimizer,
        negative_sampler='basic',
    )

    evaluator = RankBasedEvaluator()

    training_loop.train(
        num_epochs=num_epochs,
        batch_size=batch_size,
        triples_factory=training_set,
    )

    model_results = evaluator.evaluate(
        model=model,
        mapped_triples=testing_set.mapped_triples.to(device),
        additional_filter_triples=[
            training_set.mapped_triples.to(device),
            validation_set.mapped_triples.to(device),
        ],
        batch_size=512,
    )   

    a = model_results.to_df()
    mrr = a[(a['Metric'] == 'inverse_harmonic_mean_rank') & (a['Side'] == 'both') & (a['Rank_type'] == 'realistic')]['Value'].iloc[0]

    return(mrr)




In [12]:
study = optuna.create_study(
    direction="maximize",     
    study_name="distmult-hpo",
)


[I 2025-11-17 22:16:30,333] A new study created in memory with name: distmult-hpo


In [13]:
study.optimize(training_cycle, n_trials=20)


Training epochs on cuda:0: 100%|██████████| 10/10 [00:23<00:00,  2.39s/epoch, loss=1.54, prev_loss=1.74]
Evaluating on cuda:0: 100%|██████████| 42.1k/42.1k [01:35<00:00, 441triple/s]
[I 2025-11-17 22:18:38,498] Trial 0 finished with value: 0.004014371428638697 and parameters: {'embedding_dim': 160, 'lr': 0.46128843281964316, 'batch_size': 2048, 'margin': 1.8028536019885086, 'gamma': 0.9977866540105157}. Best is trial 0 with value: 0.004014371428638697.
Training epochs on cuda:0: 100%|██████████| 10/10 [00:06<00:00,  1.55epoch/s, loss=0.808, prev_loss=0.945]
Evaluating on cuda:0: 100%|██████████| 42.1k/42.1k [01:17<00:00, 542triple/s]
[I 2025-11-17 22:20:03,353] Trial 1 finished with value: 0.0024601854383945465 and parameters: {'embedding_dim': 128, 'lr': 0.3368216882699029, 'batch_size': 8192, 'margin': 1.5787266282149153, 'gamma': 0.9455428848641946}. Best is trial 0 with value: 0.004014371428638697.
Training epochs on cuda:0: 100%|██████████| 10/10 [00:34<00:00,  3.48s/epoch, loss=0

In [14]:
study.best_params


{'embedding_dim': 224,
 'lr': 0.24004214497084256,
 'batch_size': 1024,
 'margin': 1.6512000347916571,
 'gamma': 0.9370798552238292}